# 03. Статистическая проверка

H0: конверсии A и B равны. H1: конверсии различаются. Двусторонняя проверка, α = 0,05.

Рандомизация по пользователям: повторные попытки одного пользователя зависимы. Используется исходная кластерно-робастная дисперсия ratio-метрики. Пользователи без попыток сохраняются с нулями.

Каждый ноутбук запускается независимо: **Restart Kernel → Run All**. Расчёты импортируются из `src/ab_test`. Исходные данные не изменяются.

In [1]:
from pathlib import Path
import sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src' / 'ab_test').is_dir())
sys.path.insert(0, str(ROOT / 'src'))
import pandas as pd
from IPython.display import display
pd.set_option('display.max_columns', None)
from ab_test.data import load_data, prepare_data
users, payments = load_data()
users_clean, payments_clean, audit = prepare_data(users, payments)


In [2]:
from ab_test.metrics import aggregate_users
from ab_test.statistics import cluster_test
user_stats = aggregate_users(users_clean, payments_clean)
display(user_stats.head())
display(user_stats.groupby('group').agg(users=('user_id','count'), attempts=('attempts','sum'), successes=('successes','sum')))
statistics = cluster_test(user_stats)
display(pd.DataFrame(statistics['group_stats']).T)
display(pd.Series({k:v for k,v in statistics.items() if k != 'group_stats'}))

,user_id,group,attempts,successes
0,1001,A,1,0
1,1002,A,1,0
2,1003,A,1,0
3,1004,A,3,2
4,1005,A,2,1


,users,attempts,successes
group,,,
A,752,1309,652
B,751,1351,870


,cr,variance
A,0.498090,0.000185
B,0.643967,0.000172


cr_a               4.980901e-01
cr_b               6.439674e-01
delta              1.458773e-01
se_delta           1.889688e-02
z_stat             7.719651e+00
p_value            1.166489e-14
ci_lower           1.088401e-01
ci_upper           1.829145e-01
alpha              5.000000e-02
relative_uplift    2.928733e+01
dtype: float64

Оценка конверсии: Σsuccesses / Σattempts. Остаток пользователя: successesᵢ − CR × attemptsᵢ. Дисперсия группы: n/(n−1) × Σостатокᵢ² / (Σattempts)². SE разности — корень суммы дисперсий. Критическое значение нормального распределения вычисляется по α; в исходнике использовалось округление 1,96.